KERAS: CARICAMENTO DATI VELOCE E MODELLI AFFIDABILI

Carburante che alimenta le reti: i dati

- Efficienza e velocità tramite il caricamento asincrono
- Integrazione della normalizzazione direttamente nell'architettura del modello
- Metodologie per la divisione automatica dei dati

CARICAMENTO ASINCRONO
Oltre il caricamento sequenziale in memoria RAM
Con dataset di grandi dimensioni il caricamento in memoria può diventare lento ed inefficiente.
Keras offre strumenti per il caricamento ASINCRONO permettendo alla CPU di preparare il prossimo batch mentre la GPU sta ancora elaborando quello attuale.
I dati fluiscono come un fiume senza mai intasare la memoria del sistema

Pipeline di dati ottimizzate
Massimizzare l'uso dell'hardware
- Il caricamento asincrono permette di evitare il collo di bottiglia dell'input, dove il processore grafico rimane inattivo in attesa di dati.
- Le funzioni di Keras permettono di leggere dati da disco o via rete in parallelo all'addestramento in GPU
- L'uso di code di pre-caricamento assicura che il flusso di informazioni sia costante e privo di interuzioni.
- L'efficienza viene misurata tramite il tempo di inattività della GPU

Gestione della memoria
- Dataset Generatori: I generatori caricano solo una piccola porzione di dati alla volta, quando servono, rendendo possibile l'addestramento su milioni di immagini anche con poca memoria.
- Prefetching dei dati: questa tecnica sovrappone l'esecuzione del preprocessing alla fase di calcolo del gradiente, riducendo drasticamente i tempi totali. Zona di stoccaggio temporanea, prepara i dati un attimo prima che la GPU li richieda 
- Parallelismo I/O: usiamo tutti i thread della CPU per leggere i files dal disco

Dall'esecuzione sequenziale alla pipeline
Il paradigma della catena di montaggio
Senza asincronia, l'addestramento è come un cuoco che aspetta di tagliare le vedure prima di accendere i fornelli, perdendo tempo prezioso tra una fase e l'altra.
Con le pipeline di Keras, mentra un batch sta cuocendo nella GPU, la CPU sta già lavorando e tagliando gli ingredienti per il piatto successivo.
Nessuno aspetta nessuno e la cucina produce piatti alla massima velocità possibile


Normalizzazione integrata.
Spostare il prepocessing dentro il modello.
Tradizionalmente la normalizzazione dei dati avveniva come fase separata prima dell'addestramento. Keras permette di integrare questa logica direttamente come layer. La normalizzazione diventa un layer, un organo interno della nostra rete neurale.
Questo garantisce che la stessa identica trasformazione venga applicata in fase di addestramento, validazione e inferenza in produzione, riducendo gli errori umani.

Layer di Preprocessing
Standarizzazione e Riscaldamento
Inserire un layer di rescaling o normalization come primo strato del modello è buona cosa, trasforma il dato grezzo (esempio i pixel di una foto) in numeri piccoli e digeribili dai nostri neuroni (tipicamente tra 0 e 1). In questo modo il modello non impara solo a classificare ma impara anche a pulire ciò che riceve.
Se salvate il modello e lo passate ad un collega non dovrò preoccuparsi di riscalare i dati, il modello lo farà per lui.
- I layer di normalizzazione permettono di mappare i dati in intervalli numerici ottimali per la stabilità del gradiente senza script esterni
- Essendo parte del modello, questi layer vengono salvati insieme ai pesi, rendendo il modello pronto all'uso su dati grezzi.
- La noralizzazione può essere adattata statisticamente al dataset di addestramento tramite funzioni specifiche che calcolano media e varianz
- Il riscaldamento lineare porta i pixel delle immagini da un intervallo tipico ad uno standard compreso tra zero e uno

Vantaggi operativi
- Consistenza dei dati: eliminando la necessità di codice di preprocessing esterno, si evita la discrepanza tra il modo in cui i dati vengono normalizzati in locale e sul server. Quante volte i modelli falliscono in produzione perchè qualcuno ha dimenticato di dividere per 255? (per le immagini), con i layer integrati questo errore scompare
- Efficienza su GPU: molti layer di preprocessing possono essere eseguiti direttamente sulla scheda grafica GPU, accelerando ulteriormento la trasformazione dei dati
- Adattabilità Dinamica: Alcuni layer possono analizzare il dataset per impostare automaticamente i parametri di normalizzazione corretti senza intervento manuale.

Portabilità del modello
Il modello diventa un'entità autonoma
Il modello non è più un insieme di pesi, ma un intero sistema di elaborazione, rendendo il deploy in produzione più semplice.
Includere la normalizzazione nel modello significa che non dobbiamo più preoccuparci di come l'utente finale formatterà i dati in ingresso.
Possiamo inviare il file del modello a chiunque e questo accetterà direttamente i dati grezzi eseguendo la pulizia necessaria in modo trasparente.

DIVISIONE AUTOMATICA DEI DATASET
Gestione integrata di addestramento e validazione
Dividere i dati manualmente richiede codice extra e attenzione a non mescolare le informazioni. Keras automatizza questo proceso tramite parametri dedicati.
Utilizzando i metodi di caricamento integrati, è possibile specificare la percentuale di dati da destinare alla validazione con una singola riga di comando.

Automazioe dello Splitting
Rigore metodologico semplificato
- La divisione avviene in modo casuale o ordinato, assicurando che il modello venga testato su dati che non ha mai visto durante l'addestramento.
- L'integrazione con le pipeline asincrone permette di gestire lo split senza caricare i dati due volte in memoria.
- E' possibile impostare un seme casuale per garantire che la divisione sia riproducibile tra diversi esperimeti o riavii
La frazione di validazione rappresenta la percentuale di campioni sottratti all'addestramento per il monitoraggio.

Riproducibilità e controllo
- Seme casuale: impostare un seme assicura che ad ogni esecuzione i set di addestramento e validazione contengano gli stessi identici campioni. Ripetendo l'esperimento la divisione sarà sempre la stessa per confrontare i modelli.
- Caricamento da directory: Keras può creare automaticamente le etichette basandosi sulla struttura delle cartelle, dividendo poi i file internamente in set separati.
- Sotto-set specifici: E'possibile richiedere esplicitamente la parte di training o di validation garantendo che non vi sia sovrapposizione tra i due insiemi.

Best practices nello splitting
Evitare l'inquinamento dei dati
Un errore comune è normalizzare i dati prima di dividerli, permettendo alla media del validation set di influenzare il training set.
Utilizzando le funzioni di Keras, la divisione avviene a monte, garantendo che i set siano realmente indipendenti dal punto di vista statistico. Evita il datalake (inquinamento dei dati)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets

#1. CARICAMENTO ASINCRONO E DIVISIONE AUTOMATICA DEI DATI (punto chieve 1 e 3)
#Utilizziamo MIST come esempio di dataset

print("Caricamento dataset in corso...")
(train_images, train_labels), (test_images, test_labels) = datasets.mnist.load_data()

#Creazione di un oggetto Dataset per gestire la pipeline
#Usiamo il 20% del training per la validazione internamento durante il fit
#il metodo shuffle e batch creano la pipeline di caricamento asincrono

train_ds=tf.data.Dataset.from_tensor_slices((train_images,train_labels))
train_ds=train_ds.shuffle(10000).batch(32).prefetch(tf.data.AUTOTUNE)
#mescola i dati e li carica in batch di 32, prefetch per ottimizzare il caricamento (pensato per dataset molto grandi e complessi)

#2. COSTRUZIONE DEL MODELLO E NORMALIZZAZIONE INTEGRATA (punto chieve 2)

print("Costruzione del modello in corso...")
model = models.Sequential([
    #Layer di riscaldamento: trasforma i pixel da [0,255] a [0,1] direttamente all'interno del modello
    layers.Rescaling(1./255, input_shape=(28, 28, 1)), #normalizzazione integrata

    layers.Flatten(), # appiattisce l'immagine 28x28 in un vettore di 784 elementi
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

#3. COMPILAZIONE DEL MODELLO E DIVISIONE AUTOMATICA DEI DATI(punto chieve 4)

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy']) 

#Eseguiamo l'addestramento usanto il validation_split automatico di Keras
#Nota: validation_split funziona direttamente sui array numpy
print("Inizio addestramento...")
history=model.fit(
    train_images
    ,train_labels
    , epochs=5
    , validation_split=0.2 #divisione automatica 80% 20%
    , shuffle=True
    , batch_size=32) 

print("Addestramento completato.")

Caricamento dataset in corso...
Costruzione del modello in corso...


c:\Users\uberti\.conda\envs\ai_epicode\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Inizio addestramento...
Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9208 - loss: 0.2811 - val_accuracy: 0.9593 - val_loss: 0.1502
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9636 - loss: 0.1240 - val_accuracy: 0.9653 - val_loss: 0.1231
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9743 - loss: 0.0872 - val_accuracy: 0.9692 - val_loss: 0.1049
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9801 - loss: 0.0654 - val_accuracy: 0.9718 - val_loss: 0.0951
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9852 - loss: 0.0493 - val_accuracy: 0.9726 - val_loss: 0.0907
Addestramento completato.
